# Deep Retention — Análisis financiero de retención

Este notebook integra los artefactos ya entrenados con el motor financiero. No reentrena el MLP, no usa `Exited` para decidir y no modifica los archivos fuente.

La probabilidad corregida se combina con `AnnualNetMargin_USD_Sim` mediante:

\[
GNE=(P_f 	imes E_i 	imes M_c)-C_i
\]

Los costos, efectividades y reglas comerciales del catálogo son **supuestos académicos**, no resultados causales ni condiciones bancarias reales.

## 1. Carga de datos y componentes

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    root
    for root in [Path.cwd(), *Path.cwd().parents]
    if (root / "data" / "processed" / "clientes_limpios.csv").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.financial import (
    calcular_umbrales_comerciales,
    evaluar_cartera,
    resumir_cartera,
    validar_catalogo,
)
from src.prediction import PredictorFuga

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clientes_limpios.csv"
CATALOG_PATH = PROJECT_ROOT / "data" / "incentivos.csv"

clientes = pd.read_csv(DATA_PATH)
catalogo = pd.read_csv(CATALOG_PATH)

print("Clientes cargados:", len(clientes))
print("Columnas:", clientes.shape[1])
print("Catálogo:", CATALOG_PATH.relative_to(PROJECT_ROOT).as_posix())

## 2. Consistencia financiera y caracterización

`AnnualNetMargin_USD_Sim` es el único margen utilizado en la GNE. Las comisiones y el margen financiero no se vuelven a sumar durante la decisión, porque ya forman parte del margen neto.

`Balance` se usa solo mediante percentiles internos: su unidad monetaria no está declarada y no se convierte a USD.

In [ ]:
margen_recalculado = (
    clientes["AnnualCommissionIncome_USD_Sim"]
    + clientes["AnnualFinancialMargin_USD_Sim"]
    - clientes["AnnualOperatingCosts_USD_Sim"]
)
diferencia_margen = (
    margen_recalculado - clientes["AnnualNetMargin_USD_Sim"]
).abs()

print("Diferencia máxima del margen (USD):", round(diferencia_margen.max(), 4))
print(
    "Registros consistentes con tolerancia de 0,02 USD:",
    int(diferencia_margen.le(0.02).sum()),
)
print(
    "Clientes con margen no positivo:",
    int(clientes["AnnualNetMargin_USD_Sim"].le(0).sum()),
)

variables_perfil = [
    "Balance",
    "AnnualCommissionIncome_USD_Sim",
    "MonthlyTransactions_Sim",
    "MonthlyTransactionVolume_USD_Sim",
    "AnnualNetMargin_USD_Sim",
]
display(clientes[variables_perfil].describe().T)

## 3. Catálogo y umbrales comerciales

Las condiciones se evalúan antes de comparar la GNE:

- bonificación: actividad transaccional desde el percentil 50;
- exoneración: comisiones anuales desde el percentil 50;
- tasa preferencial: `Balance` desde el percentil 75 y volumen mensual desde el percentil 50;
- gestor personalizado: `Balance` desde el percentil 90 y volumen mensual desde el percentil 75.

Los mínimos de margen, costos y efectividades también son supuestos académicos versionados en el catálogo.

In [ ]:
catalogo_validado = validar_catalogo(catalogo)
umbrales = calcular_umbrales_comerciales(clientes, catalogo_validado)

filas_umbrales = []
for regla, valores in umbrales.items():
    for percentil, valor in valores.items():
        filas_umbrales.append({
            "regla": regla,
            "percentil": percentil,
            "umbral_observado": valor,
        })

display(catalogo_validado)
display(pd.DataFrame(filas_umbrales))

## 4. Predicción ajustada por lotes

La corrección por pesos de clase es aproximada: revierte el cambio de prior inducido por `class_weight`, pero no equivale a una calibración aprendida con datos independientes.

In [ ]:
predictor = PredictorFuga()
probabilidades = predictor.predecir_lote(clientes)

probabilidades_individuales = np.array([
    predictor.predecir(clientes.iloc[[position]])
    for position in range(10)
])

print("Artefacto compatible:", predictor.artifact_id)
print("Probabilidades generadas:", len(probabilidades))
print("Mínimo:", round(float(probabilidades.min()), 6))
print("Máximo:", round(float(probabilidades.max()), 6))
print("Promedio:", round(float(probabilidades.mean()), 6))
print(
    "Equivalencia individual/lote (primeros 10):",
    bool(np.allclose(probabilidades_individuales, probabilidades[:10], rtol=1e-6, atol=1e-7)),
)

## 5. Decisiones consolidadas de la cartera

In [ ]:
resultados = evaluar_cartera(
    clientes,
    probabilidades,
    catalogo_validado,
    umbrales,
)
resumen = resumir_cartera(resultados)

resumen_df = pd.DataFrame({
    "Indicador": list(resumen),
    "Valor": list(resumen.values()),
})
distribucion = (
    resultados.loc[resultados["decision"].eq("Retener"), "incentivo"]
    .value_counts()
    .rename_axis("Incentivo")
    .reset_index(name="Clientes")
)

deficitarios_retenidos = (
    resultados["decision"].eq("Retener")
    & resultados["margen_anual_usd"].le(0)
).sum()

print("Clientes deficitarios con gasto:", int(deficitarios_retenidos))
print(
    "Todas las retenciones tienen GNE positiva:",
    bool(resultados.loc[resultados["decision"].eq("Retener"), "gne"].gt(0).all()),
)
display(resumen_df)
display(distribucion)
display(
    resultados.drop(columns="alternativas").head(10)
)

## 6. Comparación con la primera implementación

In [ ]:
referencia_anterior = {
    "clientes_retener": 2319,
    "gne_total_usd": 161651.15,
}

comparacion = pd.DataFrame({
    "Indicador": ["Clientes a retener", "GNE total proyectada (USD)"],
    "Primera implementación": [
        referencia_anterior["clientes_retener"],
        referencia_anterior["gne_total_usd"],
    ],
    "Reglas comerciales nuevas": [
        resumen["clientes_retener"],
        round(resumen["gne_total_usd"], 2),
    ],
})
comparacion["Diferencia"] = (
    comparacion["Reglas comerciales nuevas"]
    - comparacion["Primera implementación"]
)
display(comparacion)

## 7. Conclusiones

- Se analizaron 10.000 clientes y los 10.000 márgenes cumplen la identidad financiera con tolerancia de 0,02 USD.
- Las nuevas condiciones comerciales recomiendan **retener a 1.214 clientes**: 367 con bonificación transaccional, 536 con exoneración de comisiones, 297 con tasa preferencial y 14 con gestor personalizado.
- **3.602 clientes** tienen margen no positivo y se clasifican como `Salida permitida`; ninguno recibe gasto. Otros **5.184 clientes rentables** quedan en `No intervención` porque no existe una alternativa simultáneamente elegible y con GNE positiva.
- La inversión académica proyectada es **USD 66.640,00**, el valor rescatable estimado es **USD 156.739,69** y la GNE total es **USD 90.099,69**.
- Frente a la primera implementación, se recomiendan 1.105 intervenciones menos y la GNE proyectada disminuye USD 71.551,46. La diferencia se debe al filtrado por actividad transaccional, comisiones, volumen y percentiles relativos de `Balance`; no se forzaron los resultados anteriores.
- Estas cifras son una simulación académica sobre una cartera que incluye registros usados durante el entrenamiento. No representan ganancias realizadas ni una evaluación predictiva independiente.